
# ECF 2021 — Generador dinámico de `ECF_DataDictionary.xlsx`

Este notebook sustituye a la versión anterior y utiliza únicamente librerías habituales:

- `pandas`
- `numpy`
- `openpyxl`
- librerías estándar de Python

## Estructura de carpetas esperada

El notebook debe guardarse en:

```text
../Scripts/
```

Los archivos de entrada deben encontrarse en:

```text
../Data/
```

El Excel generado se guardará también en:

```text
../Data/ECF_DataDictionary.xlsx
```

## Código de colores

- **Verde:** variable original conservada directamente en el dataset final.
- **Amarillo:** variable original utilizada dentro de una multirrespuesta o indicador derivado.
- **Rojo:** variable original que no aparece ni participa en los indicadores documentados del dataset final.


In [1]:

from pathlib import Path
from datetime import datetime
import json
import re

import numpy as np
import pandas as pd

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.formatting.rule import FormulaRule

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)


## 1. Configuración de rutas

In [2]:

# El notebook está previsto para ejecutarse desde la carpeta Scripts.
SCRIPTS_DIR = Path.cwd().resolve()
DATA_DIR = (SCRIPTS_DIR / "../Data").resolve()

RAW_PATH = DATA_DIR / "2026-07-20_ECF_2021_01_raw.dta"
FINAL_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset.csv"
INDICATOR_DICT_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset_Diccionario.csv"
MASTER_NOTEBOOK_PATH = SCRIPTS_DIR / "2026-07-20_ECF_2021_01_Create_Master_Dataset.ipynb"

OUTPUT_PATH = DATA_DIR / "ECF_DataDictionary.xlsx"

required_files = {
    "Raw Stata": RAW_PATH,
    "Dataset final": FINAL_PATH,
    "Diccionario de indicadores": INDICATOR_DICT_PATH,
    "Notebook de creación": MASTER_NOTEBOOK_PATH,
}

missing = {name: path for name, path in required_files.items() if not path.exists()}

if missing:
    print("No se han encontrado estos archivos:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    raise FileNotFoundError(
        "Revisa que el notebook esté dentro de Scripts y que los datos estén dentro de Data."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Scripts : {SCRIPTS_DIR}")
print(f"Data    : {DATA_DIR}")
print(f"Salida  : {OUTPUT_PATH}")


Scripts : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Scripts
Data    : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data
Salida  : /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/ECF_DataDictionary.xlsx


## 2. Lectura dinámica del estado actual

In [3]:

stata_reader = pd.io.stata.StataReader(RAW_PATH)

variable_labels = stata_reader.variable_labels()
raw_variables = list(variable_labels.keys())

# Nombre del conjunto de etiquetas asociado a cada variable.
value_label_names = list(stata_reader._lbllist)

# Diccionario: nombre del conjunto de etiquetas -> {código: etiqueta}
value_label_sets = stata_reader.value_labels()

# Solo necesitamos la cabecera del CSV final.
final_columns = list(pd.read_csv(FINAL_PATH, nrows=0).columns)

indicator_dictionary = pd.read_csv(INDICATOR_DICT_PATH)
indicator_dictionary.columns = [
    str(column).strip() for column in indicator_dictionary.columns
]

required_indicator_columns = {"Variable", "Definición"}
if not required_indicator_columns.issubset(indicator_dictionary.columns):
    raise ValueError(
        "El diccionario de indicadores debe contener las columnas "
        "'Variable' y 'Definición'."
    )

indicator_descriptions = dict(
    zip(
        indicator_dictionary["Variable"].astype(str),
        indicator_dictionary["Definición"].astype(str),
    )
)

print(f"Variables raw detectadas  : {len(raw_variables)}")
print(f"Variables finales detectadas: {len(final_columns)}")
print(f"Indicadores documentados  : {len(indicator_dictionary)}")


Variables raw detectadas  : 429
Variables finales detectadas: 255
Indicadores documentados  : 35


## 3. Mapa de trazabilidad de indicadores

In [4]:
INDICATOR_SOURCES = {'n_canales_uso_banco': ['b0110a', 'b0110b', 'b0110c', 'b0110d', 'b0110e', 'b0110f'], 'usa_banca_digital': ['b0110d', 'b0110e'], 'pref_banca_digital': ['b0120d', 'b0120e'], 'usa_pago_digital': ['b0130a', 'b0130b'], 'n_productos_financieros': ['b0301', 'b0302', 'b0303', 'b0304', 'b0305', 'b0306', 'b0307', 'b0308', 'b0309', 'b0310', 'b0312'], 'tiene_vehiculo_ahorro': ['b0302', 'b0303', 'b0304', 'b0305', 'b0308', 'b0312'], 'tiene_exposicion_credito': ['b0301', 'b0306', 'b0307'], 'tiene_seguro': ['b0309', 'b0310'], 'ahorra_12m': ['b1000a', 'b1000b', 'b1000c', 'b1000d', 'b1000e', 'b1000f', 'b1000g', 'b1000h', 'b1000i', 'b1000j'], 'n_vehiculos_ahorro': ['b1000a', 'b1000b', 'b1000c', 'b1000d', 'b1000e', 'b1000f', 'b1000g', 'b1000h', 'b1000i'], 'ahorro_formal': ['b1000b', 'b1000c', 'b1000g', 'b1000h'], 'ahorro_informal': ['b1000a', 'b1000d', 'b1000e', 'b1000f', 'b1000i'], 'n_fuentes_ingreso': ['c0200a', 'c0200b', 'c0200c', 'c0200d', 'c0200e', 'c0200f', 'c0200g', 'c0200h', 'c0200i', 'c0200j', 'c0200k', 'c0400a', 'c0400b', 'c0400c', 'c0400d', 'c0400e', 'c0400f', 'c0400g', 'c0400h', 'c0400i', 'c0400j', 'c0400k', 'c0400l', 'c0400m', 'c0400n', 'c0600a', 'c0600b', 'c0600c', 'c0600d', 'c0600e', 'c0600f', 'c0600g', 'c0600h', 'c0600i', 'c0600j', 'c0600k'], 'ingreso_por_activos': ['c0200f', 'c0200g', 'c0200h', 'c0400h', 'c0400i', 'c0400j', 'c0400k', 'c0400m', 'c0600f', 'c0600g', 'c0600h'], 'dependencia_ingresos_familiares': ['c0200d', 'c0200e', 'c0200i', 'c0400c', 'c0400d', 'c0400e', 'c0400f', 'c0400l', 'c0600d', 'c0600e', 'c0600i'], 'score_disciplina_financiera': ['d0101', 'd0104', 'd0106', 'd0113'], 'score_planificacion_financiera': ['d0102', 'd0103', 'd0107', 'd0108', 'd0116'], 'score_preocupacion_financiera': ['d0109', 'd0110', 'd0111', 'd0114', 'd0117'], 'puntos_destino_consumo': ['d0610a', 'd0610b'], 'puntos_destino_ahorro': ['d0610c'], 'puntos_destino_deuda': ['d0610d'], 'score_alfabetizacion_financiera': ['e0600', 'e0900', 'e1003'], 'score_numeracy_financiera': ['e0500', 'e0700', 'e0800'], 'score_comprension_riesgo': ['e1001', 'e1003', 'e1101'], 'score_competencia_economica': ['e1701', 'e1702', 'e1800'], 'score_conocimiento_productos': ['e1002', 'e1200'], 'barrera_acceso_vivienda': ['i0200d', 'i0200e', 'i0200h', 'i0520d', 'i0520e', 'i0520f'], 'expectativa_subida_precio_vivienda': ['i0400a', 'i0400b', 'i0400c', 'i0400d', 'i0400e'], 'n_dificultades_compra_vivienda': ['i0520a', 'i0520b', 'i0520c', 'i0520d', 'i0520e', 'i0520f', 'i0520g'], 'financiacion_ahorros_activos': ['j0300a', 'j0300b'], 'financiacion_credito_informal': ['j0300c', 'j0300d', 'j0300l'], 'financiacion_credito_formal': ['j0300e', 'j0300f', 'j0300g', 'j0300h', 'j0300i', 'j0300j', 'j0300k'], 'financiacion_estres_pago': ['j0300m', 'j0300n'], 'restriccion_acceso_credito': ['j0900a', 'j0900b', 'j0900c'], 'score_fragilidad_financiera': ['j0200', 'j1000', 'j1201', 'j0900a', 'j0900b', 'j0900c']}


El mapa anterior identifica qué variables originales se utilizaron para construir cada indicador.

El resto del notebook sigue siendo dinámico:

- vuelve a leer todas las columnas del `.dta`;
- vuelve a leer todas las columnas del CSV final;
- comprueba qué indicadores continúan presentes;
- marca como no documentado cualquier indicador nuevo que no aparezca en este mapa.


## 4. Funciones auxiliares

In [5]:

def get_block(variable):
    block_code = str(variable).lower()[:1]

    blocks = {
        "a": "A - Demografía y situación laboral",
        "b": "B - Cartera, banca, productos y ahorro",
        "c": "C - Fuentes de renta",
        "d": "D - Actitudes ante el ahorro",
        "e": "E - Competencias financieras",
        "f": "F - Decisiones del hogar",
        "i": "I - Vivienda principal",
        "j": "J - Gasto y fragilidad financiera",
        "k": "K - Persona con mayor conocimiento financiero",
    }

    return blocks.get(block_code, "Variables auxiliares o de identificación")


def is_multiresponse_option(variable):
    """Ejemplo: b0110a, b0110b, b0110c..."""
    return bool(re.match(r"^[a-z]\d{4}[a-z]$", str(variable).lower()))


def get_multiresponse_parent(variable):
    """Ejemplo: b0110a -> b0110x."""
    match = re.match(r"^([a-z]\d{4})([a-z])$", str(variable).lower())
    return f"{match.group(1)}x" if match else ""


def format_value_labels(label_name):
    if not label_name:
        return ""

    mapping = value_label_sets.get(label_name, {})

    if not mapping:
        return ""

    return " | ".join(
        f"{code} = {label}"
        for code, label in mapping.items()
    )


def safe_join(values):
    return ", ".join(str(value) for value in values if str(value).strip())


def add_excel_table(ws, table_name):
    if ws.max_row < 2 or ws.max_column < 1:
        return

    reference = (
        f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    )

    table = Table(displayName=table_name, ref=reference)
    style = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=False,
        showColumnStripes=False,
    )
    table.tableStyleInfo = style
    ws.add_table(table)


def set_column_widths(ws, widths):
    for column, width in widths.items():
        ws.column_dimensions[column].width = width


def apply_standard_sheet_style(ws):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    thin_gray = Side(style="thin", color="D9E2F3")

    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )
            cell.border = Border(
                bottom=thin_gray,
            )


def style_header(ws):
    header_fill = PatternFill("solid", fgColor="24557A")
    header_font = Font(color="FFFFFF", bold=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    ws.row_dimensions[1].height = 32


## 5. Construcción del diccionario de variables raw

In [6]:

source_to_indicators = {}

for indicator, source_variables in INDICATOR_SOURCES.items():
    for source_variable in source_variables:
        source_to_indicators.setdefault(
            source_variable,
            []
        ).append(indicator)

raw_rows = []

for position, variable in enumerate(raw_variables):
    direct_inclusion = variable in final_columns
    target_indicators = sorted(
        source_to_indicators.get(variable, [])
    )
    multiresponse = is_multiresponse_option(variable)

    if direct_inclusion:
        status = "Conservada directamente"
        comparison = "Sí"
        final_target = variable
        color_code = "VERDE"

    elif target_indicators:
        comparison = "Indirectamente"
        final_target = safe_join(target_indicators)
        color_code = "AMARILLO"

        if multiresponse:
            status = "Integrada desde multirrespuesta"
        else:
            status = "Integrada en indicador derivado"

    else:
        status = "No incluida en el dataset final"
        comparison = "No"
        final_target = ""
        color_code = "ROJO"

    label_name = (
        value_label_names[position]
        if position < len(value_label_names)
        else ""
    )

    indicator_description = " | ".join(
        indicator_descriptions.get(indicator, "")
        for indicator in target_indicators
        if indicator_descriptions.get(indicator, "")
    )

    raw_rows.append({
        "Orden_raw": position + 1,
        "Variable_raw": variable,
        "Bloque_ECF": get_block(variable),
        "Etiqueta_pregunta_Stata": variable_labels.get(variable, ""),
        "Tipo_origen": (
            "Opción de multirrespuesta"
            if multiresponse
            else "Variable individual"
        ),
        "Pregunta_multirrespuesta_padre": (
            get_multiresponse_parent(variable)
            if multiresponse
            else ""
        ),
        "Etiquetas_de_valores": format_value_labels(label_name),
        "Aparece_en_dataset_final_65": comparison,
        "Estado_comparación": status,
        "Variable_final_o_indicador": final_target,
        "Descripción_indicador_final": indicator_description,
        "Código_color": color_code,
        "Fuente": "Metadatos del fichero Stata ECF 2021",
    })

raw_dictionary = pd.DataFrame(raw_rows)

display(raw_dictionary.head(10))


,Orden_raw,Variable_raw,Bloque_ECF,Etiqueta_pregunta_Stata,Tipo_origen,Pregunta_multirrespuesta_padre,Etiquetas_de_valores,Aparece_en_dataset_final_65,Estado_comparación,Variable_final_o_indicador,Descripción_indicador_final,Código_color,Fuente
0,1,a01,A - Demografía y situación laboral,a01: anio de la entrevista,Variable individual,,,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
1,2,a02,A - Demografía y situación laboral,a02: mes de la entrevista,Variable individual,,,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
2,3,a0000,A - Demografía y situación laboral,"a0000: es requisito que pregunte su genero, es vd. hombre o mujer?",Variable individual,,0 = Mujer | 1 = Hombre,Sí,Conservada directamente,a0000,,VERDE,Metadatos del fichero Stata ECF 2021
3,4,a0400,A - Demografía y situación laboral,a0400: en que anio nacio?,Variable individual,,-99 = No contesta | -97 = No sabe,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
4,5,a04,A - Demografía y situación laboral,a04: edad calculada,Variable individual,,-98 = No ha lugar (no aplica filtro),Sí,Conservada directamente,a04,,VERDE,Metadatos del fichero Stata ECF 2021
5,6,a0800,A - Demografía y situación laboral,a0800: me podria decir su edad aproximada?,Variable individual,,-98 = No ha lugar (no aplica filtro),No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
6,7,a0100,A - Demografía y situación laboral,a0100: en que pais nacio?,Variable individual,,-99 = No contesta | -97 = No sabe | 0 = Otro país | 1 = España,Sí,Conservada directamente,a0100,,VERDE,Metadatos del fichero Stata ECF 2021
7,8,a0320,A - Demografía y situación laboral,a0320: nacio alguno de sus padres fuera de espania?,Variable individual,,-99 = No contesta | -97 = No sabe | 0 = No | 1 = Sí,Sí,Conservada directamente,a0320,,VERDE,Metadatos del fichero Stata ECF 2021
8,9,a1030,A - Demografía y situación laboral,a1030: cual es su estado civil actual?,Variable individual,,"-99 = No contesta | -97 = No sabe | 1 = Soltero | 2 = Casado | 3 = Pareja de hecho | 4 = Separado, pero legalmente aún casado o con pareja de hecho | 5 = Divorciado | 6 = Viudo",Sí,Conservada directamente,a1030,,VERDE,Metadatos del fichero Stata ECF 2021
9,10,a1040,A - Demografía y situación laboral,a1040: cual es su regimen economico matrimonial?,Variable individual,,-99 = No contesta | -98 = No ha lugar (no aplica filtro) | -97 = No sabe | 1 = Separación de bienes | 2 = Gananciales | 3 = Otros (especificar),No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021


## 6. Diccionario del dataset final

In [7]:

final_rows = []

for position, variable in enumerate(final_columns, start=1):
    if variable in variable_labels:
        variable_type = "Variable original conservada"
        description = variable_labels.get(variable, "")
        source_variables = variable
        source_count = 1
        documentation_status = "Documentada"

    else:
        variable_type = "Indicador derivado"
        source_list = INDICATOR_SOURCES.get(variable, [])
        description = indicator_descriptions.get(variable, "")
        source_variables = safe_join(source_list)
        source_count = len(source_list)

        if variable in INDICATOR_SOURCES:
            documentation_status = "Documentada"
        else:
            documentation_status = "Revisar: indicador sin mapa de fuentes"

    final_rows.append({
        "Orden_final": position,
        "Variable_final": variable,
        "Tipo": variable_type,
        "Descripción": description,
        "Variables_originales_fuente": source_variables,
        "N_fuentes": source_count,
        "Estado_documentación": documentation_status,
    })

final_dictionary = pd.DataFrame(final_rows)

display(final_dictionary)


,Orden_final,Variable_final,Tipo,Descripción,Variables_originales_fuente,N_fuentes,Estado_documentación
0,1,ccaaf,Variable original conservada,ccaaf: codigos de comunidad autonoma de residencia,ccaaf,1,Documentada
1,2,a0000,Variable original conservada,"a0000: es requisito que pregunte su genero, es vd. hombre o mujer?",a0000,1,Documentada
2,3,a04,Variable original conservada,a04: edad calculada,a04,1,Documentada
3,4,a0100,Variable original conservada,a0100: en que pais nacio?,a0100,1,Documentada
4,5,a0320,Variable original conservada,a0320: nacio alguno de sus padres fuera de espania?,a0320,1,Documentada
...,...,...,...,...,...,...,...
250,251,financiacion_credito_informal,Indicador derivado,"Cubrió el déficit mediante familia, adelantos o proveedores.","j0300c, j0300d, j0300l",3,Documentada
251,252,financiacion_credito_formal,Indicador derivado,Cubrió el déficit mediante crédito o financiación formal.,"j0300e, j0300f, j0300g, j0300h, j0300i, j0300j, j0300k",7,Documentada
252,253,financiacion_estres_pago,Indicador derivado,Utilizó descubierto no autorizado o retrasó pagos.,"j0300m, j0300n",2,Documentada
253,254,restriccion_acceso_credito,Indicador derivado,"Rechazo, concesión parcial o autoexclusión crediticia.","j0900a, j0900b, j0900c",3,Documentada


## 7. Mapa de indicadores y variables eliminadas

In [8]:

indicator_rows = []

for indicator in final_columns:
    if indicator in variable_labels:
        continue

    sources = INDICATOR_SOURCES.get(indicator, [])

    indicator_rows.append({
        "Indicador_final": indicator,
        "Definición": indicator_descriptions.get(indicator, ""),
        "N_variables_fuente": len(sources),
        "Variables_fuente": safe_join(sources),
        "Incluido_actualmente": "Sí",
        "Estado_documentación": (
            "Documentado"
            if indicator in INDICATOR_SOURCES
            else "Revisar"
        ),
    })

indicator_mapping = pd.DataFrame(indicator_rows)

eliminated_variables = raw_dictionary.loc[
    raw_dictionary["Estado_comparación"]
    == "No incluida en el dataset final"
].copy()

display(indicator_mapping)
display(eliminated_variables.head(20))


,Indicador_final,Definición,N_variables_fuente,Variables_fuente,Incluido_actualmente,Estado_documentación
0,a1420x,,0,,Sí,Revisar
1,b0110x,,0,,Sí,Revisar
2,b0120x,,0,,Sí,Revisar
3,b0130x,,0,,Sí,Revisar
4,b1100x,,0,,Sí,Revisar
...,...,...,...,...,...,...
57,financiacion_credito_informal,"Cubrió el déficit mediante familia, adelantos o proveedores.",3,"j0300c, j0300d, j0300l",Sí,Documentado
58,financiacion_credito_formal,Cubrió el déficit mediante crédito o financiación formal.,7,"j0300e, j0300f, j0300g, j0300h, j0300i, j0300j, j0300k",Sí,Documentado
59,financiacion_estres_pago,Utilizó descubierto no autorizado o retrasó pagos.,2,"j0300m, j0300n",Sí,Documentado
60,restriccion_acceso_credito,"Rechazo, concesión parcial o autoexclusión crediticia.",3,"j0900a, j0900b, j0900c",Sí,Documentado


,Orden_raw,Variable_raw,Bloque_ECF,Etiqueta_pregunta_Stata,Tipo_origen,Pregunta_multirrespuesta_padre,Etiquetas_de_valores,Aparece_en_dataset_final_65,Estado_comparación,Variable_final_o_indicador,Descripción_indicador_final,Código_color,Fuente
0,1,a01,A - Demografía y situación laboral,a01: anio de la entrevista,Variable individual,,,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
1,2,a02,A - Demografía y situación laboral,a02: mes de la entrevista,Variable individual,,,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
3,4,a0400,A - Demografía y situación laboral,a0400: en que anio nacio?,Variable individual,,-99 = No contesta | -97 = No sabe,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
5,6,a0800,A - Demografía y situación laboral,a0800: me podria decir su edad aproximada?,Variable individual,,-98 = No ha lugar (no aplica filtro),No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
9,10,a1040,A - Demografía y situación laboral,a1040: cual es su regimen economico matrimonial?,Variable individual,,-99 = No contesta | -98 = No ha lugar (no aplica filtro) | -97 = No sabe | 1 = Separación de bienes | 2 = Gananciales | 3 = Otros (especificar),No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
12,13,a1200,A - Demografía y situación laboral,a1200: cual es el contenido del titulo universitario que ha obtenido?,Variable individual,,"-99 = No contesta | -98 = No ha lugar (no aplica filtro) | -97 = No sabe | 1 = a. Ingeniería y tecnología (Arquitectura, Electrónica, Mecánica…) | 2 = b. Ciencias de la salud (...",No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
13,14,a1300,A - Demografía y situación laboral,a1300: cual es el contenido del titulo de formacion profesional que ha obtenido?,Variable individual,,"-99 = No contesta | -98 = No ha lugar (no aplica filtro) | -97 = No sabe | 1 = a. Agricultura, Especialidad Marítimo-pesquera, Minería, Energía y agua | 2 = b. Industria (Alime...",No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
16,17,a1420a,A - Demografía y situación laboral,a1420a: ha recibido formación financiera de otra manera?: a. colegio,Opción de multirrespuesta,a1420x,-99 = No contesta | -97 = No sabe | 0 = No mencionado | 1 = Mencionado,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
17,18,a1420b,A - Demografía y situación laboral,a1420b: ha recibido formación financiera de otra manera?: b. universidad,Opción de multirrespuesta,a1420x,-99 = No contesta | -97 = No sabe | 0 = No mencionado | 1 = Mencionado,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021
18,19,a1420c,A - Demografía y situación laboral,a1420c: ha recibido formación financiera de otra manera?: c. trabajo,Opción de multirrespuesta,a1420x,-99 = No contesta | -97 = No sabe | 0 = No mencionado | 1 = Mencionado,No,No incluida en el dataset final,,,ROJO,Metadatos del fichero Stata ECF 2021


## 8. Resumen de la comparación

In [9]:

summary = pd.DataFrame([
    {
        "Métrica": "Variables en fichero raw Stata",
        "Valor": len(raw_variables),
    },
    {
        "Métrica": "Variables en dataset final",
        "Valor": len(final_columns),
    },
    {
        "Métrica": "Variables raw conservadas directamente",
        "Valor": int(
            (
                raw_dictionary["Estado_comparación"]
                == "Conservada directamente"
            ).sum()
        ),
    },
    {
        "Métrica": "Variables raw integradas desde multirrespuesta",
        "Valor": int(
            (
                raw_dictionary["Estado_comparación"]
                == "Integrada desde multirrespuesta"
            ).sum()
        ),
    },
    {
        "Métrica": "Variables raw integradas en otros indicadores",
        "Valor": int(
            (
                raw_dictionary["Estado_comparación"]
                == "Integrada en indicador derivado"
            ).sum()
        ),
    },
    {
        "Métrica": "Variables raw no incluidas",
        "Valor": len(eliminated_variables),
    },
    {
        "Métrica": "Indicadores derivados presentes",
        "Valor": int(
            (
                final_dictionary["Tipo"]
                == "Indicador derivado"
            ).sum()
        ),
    },
    {
        "Métrica": "Indicadores finales sin mapa de fuentes",
        "Valor": int(
            (
                final_dictionary["Estado_documentación"]
                == "Revisar: indicador sin mapa de fuentes"
            ).sum()
        ),
    },
])

display(summary)


,Métrica,Valor
0,Variables en fichero raw Stata,429
1,Variables en dataset final,255
2,Variables raw conservadas directamente,193
3,Variables raw integradas desde multirrespuesta,83
4,Variables raw integradas en otros indicadores,0
5,Variables raw no incluidas,153
6,Indicadores derivados presentes,62
7,Indicadores finales sin mapa de fuentes,27


## 9. Generación del Excel con `openpyxl`

In [10]:

GREEN_FILL = PatternFill("solid", fgColor="C6EFCE")
YELLOW_FILL = PatternFill("solid", fgColor="FFF2CC")
RED_FILL = PatternFill("solid", fgColor="F4CCCC")
BLUE_FILL = PatternFill("solid", fgColor="DDEBF7")
DARK_BLUE_FILL = PatternFill("solid", fgColor="163A5F")
GRAY_FILL = PatternFill("solid", fgColor="E7E6E6")

WHITE_BOLD_FONT = Font(color="FFFFFF", bold=True)
TITLE_FONT = Font(color="FFFFFF", bold=True, size=16)


def write_dataframe(ws, dataframe):
    ws.append(list(dataframe.columns))

    for row in dataframe.itertuples(index=False, name=None):
        ws.append(list(row))

    style_header(ws)
    apply_standard_sheet_style(ws)


def color_raw_rows(ws, dataframe):
    status_column = (
        list(dataframe.columns).index("Estado_comparación") + 1
    )

    for excel_row in range(2, ws.max_row + 1):
        status = ws.cell(
            row=excel_row,
            column=status_column,
        ).value

        if status == "Conservada directamente":
            fill = GREEN_FILL
        elif status in {
            "Integrada desde multirrespuesta",
            "Integrada en indicador derivado",
        }:
            fill = YELLOW_FILL
        else:
            fill = RED_FILL

        for cell in ws[excel_row]:
            cell.fill = fill


def color_final_rows(ws, dataframe):
    type_column = list(dataframe.columns).index("Tipo") + 1
    documentation_column = (
        list(dataframe.columns).index("Estado_documentación") + 1
    )

    for excel_row in range(2, ws.max_row + 1):
        variable_type = ws.cell(
            row=excel_row,
            column=type_column,
        ).value

        documentation_status = ws.cell(
            row=excel_row,
            column=documentation_column,
        ).value

        fill = (
            GREEN_FILL
            if variable_type == "Variable original conservada"
            else YELLOW_FILL
        )

        if documentation_status != "Documentada":
            fill = RED_FILL

        for cell in ws[excel_row]:
            cell.fill = fill


workbook = Workbook()

# Eliminar hoja creada por defecto.
default_sheet = workbook.active
workbook.remove(default_sheet)

# ---------------------------------------------------------
# Hoja Resumen
# ---------------------------------------------------------
ws = workbook.create_sheet("Resumen")

ws.merge_cells("A1:F1")
ws["A1"] = "ECF 2021 - Diccionario dinámico y trazabilidad"
ws["A1"].fill = DARK_BLUE_FILL
ws["A1"].font = TITLE_FONT
ws["A1"].alignment = Alignment(
    horizontal="left",
    vertical="center",
)
ws.row_dimensions[1].height = 28

ws["A3"] = "Métrica"
ws["B3"] = "Valor"

for cell in ws[3]:
    if cell.column <= 2:
        cell.fill = PatternFill("solid", fgColor="24557A")
        cell.font = WHITE_BOLD_FONT

for row_number, row in enumerate(
    summary.itertuples(index=False),
    start=4,
):
    ws.cell(row=row_number, column=1, value=row.Métrica)
    ws.cell(row=row_number, column=2, value=row.Valor)

legend_start = len(summary) + 6

legend_headers = [
    "Estado",
    "Interpretación",
    "Color",
    "Fuente raw",
    "Destino final",
    "Actualización",
]

for column_number, value in enumerate(
    legend_headers,
    start=1,
):
    cell = ws.cell(
        row=legend_start,
        column=column_number,
        value=value,
    )
    cell.fill = PatternFill("solid", fgColor="24557A")
    cell.font = WHITE_BOLD_FONT

legend_rows = [
    [
        "Conservada directamente",
        "La variable raw aparece con el mismo nombre.",
        "Verde",
        "Sí",
        "Misma variable",
        "Dinámica",
    ],
    [
        "Integrada desde multirrespuesta",
        "Opción a/b/c... utilizada en un indicador.",
        "Amarillo",
        "Sí",
        "Indicador derivado",
        "Dinámica",
    ],
    [
        "Integrada en indicador derivado",
        "Variable individual utilizada en un score o indicador.",
        "Amarillo",
        "Sí",
        "Indicador derivado",
        "Dinámica",
    ],
    [
        "No incluida",
        "No aparece ni se usa en el mapa documentado.",
        "Rojo",
        "Sí",
        "Sin destino",
        "Dinámica",
    ],
]

legend_fills = [
    GREEN_FILL,
    YELLOW_FILL,
    YELLOW_FILL,
    RED_FILL,
]

for row_offset, (legend_row, fill) in enumerate(
    zip(legend_rows, legend_fills),
    start=1,
):
    target_row = legend_start + row_offset

    for column_number, value in enumerate(
        legend_row,
        start=1,
    ):
        cell = ws.cell(
            row=target_row,
            column=column_number,
            value=value,
        )
        cell.fill = fill
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

notes_start = legend_start + len(legend_rows) + 3

ws.cell(
    row=notes_start,
    column=1,
    value="Archivos utilizados",
).fill = BLUE_FILL
ws.cell(
    row=notes_start,
    column=1,
).font = Font(bold=True)

metadata_rows = [
    ["Fichero raw", RAW_PATH.name],
    ["Dataset final", FINAL_PATH.name],
    ["Diccionario de indicadores", INDICATOR_DICT_PATH.name],
    ["Notebook de creación", MASTER_NOTEBOOK_PATH.name],
    ["Fecha de generación", datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    [
        "Nota metodológica",
        (
            "Las etiquetas de pregunta y valores proceden de los "
            "metadatos del fichero Stata. El PDF del cuestionario "
            "debe utilizarse como referencia para filtros y "
            "redacción completa."
        ),
    ],
]

for row_offset, values in enumerate(
    metadata_rows,
    start=1,
):
    target_row = notes_start + row_offset
    ws.cell(row=target_row, column=1, value=values[0])
    ws.cell(row=target_row, column=2, value=values[1])

ws.freeze_panes = "A3"
set_column_widths(
    ws,
    {
        "A": 40,
        "B": 75,
        "C": 16,
        "D": 18,
        "E": 24,
        "F": 18,
    },
)

for row in ws.iter_rows():
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

# ---------------------------------------------------------
# Hoja Diccionario_Raw
# ---------------------------------------------------------
ws = workbook.create_sheet("Diccionario_Raw")
write_dataframe(ws, raw_dictionary)
color_raw_rows(ws, raw_dictionary)
add_excel_table(ws, "TablaDiccionarioRaw")

set_column_widths(
    ws,
    {
        "A": 10,
        "B": 16,
        "C": 34,
        "D": 62,
        "E": 26,
        "F": 27,
        "G": 65,
        "H": 20,
        "I": 34,
        "J": 48,
        "K": 62,
        "L": 15,
        "M": 44,
    },
)

ws.freeze_panes = "B2"

# ---------------------------------------------------------
# Hoja Dataset_Final_65
# ---------------------------------------------------------
ws = workbook.create_sheet("Dataset_Final_65")
write_dataframe(ws, final_dictionary)
color_final_rows(ws, final_dictionary)
add_excel_table(ws, "TablaDatasetFinal")

set_column_widths(
    ws,
    {
        "A": 12,
        "B": 34,
        "C": 30,
        "D": 65,
        "E": 85,
        "F": 14,
        "G": 34,
    },
)

# ---------------------------------------------------------
# Hoja Mapa_Indicadores
# ---------------------------------------------------------
ws = workbook.create_sheet("Mapa_Indicadores")
write_dataframe(ws, indicator_mapping)
add_excel_table(ws, "TablaMapaIndicadores")

for row in ws.iter_rows(min_row=2):
    fill = (
        YELLOW_FILL
        if row[-1].value == "Documentado"
        else RED_FILL
    )

    for cell in row:
        cell.fill = fill

set_column_widths(
    ws,
    {
        "A": 36,
        "B": 70,
        "C": 20,
        "D": 95,
        "E": 22,
        "F": 24,
    },
)

# ---------------------------------------------------------
# Hoja Variables_Eliminadas
# ---------------------------------------------------------
ws = workbook.create_sheet("Variables_Eliminadas")
write_dataframe(ws, eliminated_variables)
add_excel_table(ws, "TablaVariablesEliminadas")

for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.fill = RED_FILL

set_column_widths(
    ws,
    {
        "A": 10,
        "B": 16,
        "C": 34,
        "D": 62,
        "E": 26,
        "F": 27,
        "G": 65,
        "H": 20,
        "I": 34,
        "J": 48,
        "K": 62,
        "L": 15,
        "M": 44,
    },
)

# ---------------------------------------------------------
# Hoja Metadatos
# ---------------------------------------------------------
ws = workbook.create_sheet("Metadatos")

metadata = pd.DataFrame([
    ["Fichero raw", RAW_PATH.name],
    ["Dataset final", FINAL_PATH.name],
    ["Diccionario de indicadores", INDICATOR_DICT_PATH.name],
    ["Notebook de creación", MASTER_NOTEBOOK_PATH.name],
    ["Carpeta Data", str(DATA_DIR)],
    ["Carpeta Scripts", str(SCRIPTS_DIR)],
    ["Variables raw detectadas", len(raw_variables)],
    ["Variables finales detectadas", len(final_columns)],
    ["Fecha de generación", datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
], columns=["Campo", "Valor"])

write_dataframe(ws, metadata)
add_excel_table(ws, "TablaMetadatos")
set_column_widths(ws, {"A": 35, "B": 100})

# ---------------------------------------------------------
# Guardado
# ---------------------------------------------------------
workbook.save(OUTPUT_PATH)

print(f"✅ Excel generado correctamente:")
print(OUTPUT_PATH)


✅ Excel generado correctamente:
/Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data/ECF_DataDictionary.xlsx


## 10. Validaciones finales

In [11]:

from openpyxl import load_workbook

assert len(raw_dictionary) == len(raw_variables)
assert len(final_dictionary) == len(final_columns)
assert raw_dictionary["Variable_raw"].is_unique
assert final_dictionary["Variable_final"].is_unique
assert OUTPUT_PATH.exists()

validation_workbook = load_workbook(
    OUTPUT_PATH,
    read_only=True,
    data_only=False,
)

expected_sheets = {
    "Resumen",
    "Diccionario_Raw",
    "Dataset_Final_65",
    "Mapa_Indicadores",
    "Variables_Eliminadas",
    "Metadatos",
}

missing_sheets = (
    expected_sheets
    - set(validation_workbook.sheetnames)
)

if missing_sheets:
    raise AssertionError(
        f"Faltan hojas en el Excel: {missing_sheets}"
    )

undocumented_final_variables = final_dictionary.loc[
    final_dictionary["Estado_documentación"]
    != "Documentada",
    "Variable_final",
].tolist()

mapped_sources_not_found_in_raw = sorted({
    source
    for sources in INDICATOR_SOURCES.values()
    for source in sources
    if source not in raw_variables
})

print("Validación completada.")
print(f"Hojas: {validation_workbook.sheetnames}")
print(
    "Indicadores finales sin mapa de fuentes:",
    undocumented_final_variables,
)
print(
    "Variables mapeadas no encontradas en raw:",
    mapped_sources_not_found_in_raw,
)

display(
    raw_dictionary["Estado_comparación"]
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="N")
)


Validación completada.
Hojas: ['Resumen', 'Diccionario_Raw', 'Dataset_Final_65', 'Mapa_Indicadores', 'Variables_Eliminadas', 'Metadatos']
Indicadores finales sin mapa de fuentes: ['a1420x', 'b0110x', 'b0120x', 'b0130x', 'b1100x', 'b0720x', 'b0710x', 'b1000x', 'b1203x', 'b1204x', 'b1205x', 'b1206x', 'b1207x', 'b1208x', 'b1209x', 'b1230x', 'c0200x', 'c0600x', 'c0400x', 'i0200x', 'i0500x', 'i0520x', 'j0300x', 'j0800x', 'j0900x', 'j0910x', 'a0900x']
Variables mapeadas no encontradas en raw: []


,Estado,N
0,Conservada directamente,193
1,No incluida en el dataset final,153
2,Integrada desde multirrespuesta,83
